In [34]:
import pandas as pd
import numpy as np

df = pd.read_csv("Used_Car_Price_Prediction.csv")


In [35]:
numeric_features = [
    'yr_mfr',
    'kms_run',
    'total_owners'
]

categorical_features = [
    'fuel_type',
    'city',
    'body_type',
    'transmission',
    'make',
    'model'
]

In [36]:
target = 'sale_price'

In [37]:
df = df[df['sale_price'] > 0].copy()

In [38]:
df['body_type'] = df['body_type'].fillna(df['body_type'].mode()[0])
df['transmission'] = df['transmission'].fillna(df['transmission'].mode()[0])

In [39]:
X = df[numeric_features + categorical_features]
y = df['sale_price']

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

basic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

basic_model.fit(X_train, y_train)

y_pred = basic_model.predict(X_test)

In [43]:
from sklearn import metrics

mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = metrics.root_mean_squared_error(y_test, y_pred)
r2 = metrics.r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 59090.95032988885
MSE : 12419436989.674763
RMSE: 111442.52774266546
R2  : 0.8430237692980045


In [44]:
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print(
    "Encoded train shape:",
    basic_model.named_steps['preprocessor'].transform(X_train).shape
)

print(
    "Encoded test shape:",
    basic_model.named_steps['preprocessor'].transform(X_test).shape
)

Train shape: (5917, 9)
Test shape : (1480, 9)
Encoded train shape: (5917, 234)
Encoded test shape: (1480, 234)


In [45]:
encoder = basic_model.named_steps['preprocessor'].named_transformers_['cat']

for col, categories in zip(categorical_features, encoder.categories_):
    print(col, len(categories))

fuel_type 5
city 13
body_type 5
transmission 2
make 27
model 179


In [46]:
for col in categorical_features:
    train_values = set(X_train[col].unique())
    test_values = set(X_test[col].unique())

    unknown = test_values - train_values

    print(col, "→", unknown)

fuel_type → set()
city → set()
body_type → set()
transmission → set()
make → set()
model → {'passat', 'aria', 'nuvosport', 'linea', 'indica ev2', 'ml class'}


In [47]:
for col in categorical_features:
    train_values = set(X_train[col].unique())
    test_values = set(X_test[col].unique())

    unknown = test_values - train_values

    if unknown:
        print(col, ":", unknown)
        
        for value in unknown:
            print(value, "count:", (X_test[col] == value).sum())

model : {'passat', 'aria', 'nuvosport', 'linea', 'indica ev2', 'ml class'}
passat count: 1
aria count: 1
nuvosport count: 1
linea count: 1
indica ev2 count: 1
ml class count: 1


In [48]:
comparison = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred
})

comparison['error'] = abs(
    comparison['actual'] - comparison['predicted']
)

print(
    comparison
    .sort_values('error', ascending=False)
    .head(10)
)

       actual     predicted         error
3756  3250000  1.628848e+06  1.621152e+06
2498  2585899  1.545352e+06  1.040547e+06
2861  1917988  8.905933e+05  1.027395e+06
3748  2462277  1.510247e+06  9.520298e+05
2034  1972528  1.106934e+06  8.655942e+05
5492  1830522  1.008512e+06  8.220098e+05
7239   200000  9.935176e+05  7.935176e+05
1210  1538430  9.022556e+05  6.361744e+05
6592   554499  1.138555e+06  5.840558e+05
1565  1566099  1.042282e+06  5.238173e+05


In [49]:
for model in ['aria', 'passat', 'nuvosport', 'linea', 'indica ev2', 'ml class']:
    print("\nMODEL:", model)
    print(
        comparison[
            X_test['model'] == model
        ]
    )


MODEL: aria
      actual      predicted         error
1582  335199  310737.161479  24461.838521

MODEL: passat
      actual      predicted        error
6770  582899  585767.648203  2868.648203

MODEL: nuvosport
      actual      predicted         error
1915  407499  497500.228699  90001.228699

MODEL: linea
      actual      predicted         error
1608  257499  331388.910085  73889.910085

MODEL: indica ev2
      actual      predicted          error
6397  150000  331802.495569  181802.495569

MODEL: ml class
       actual     predicted          error
2034  1972528  1.106934e+06  865594.200568


In [50]:
old_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

print("Old encoded shape:", old_encoded.shape)

Old encoded shape: (7397, 240)


In [51]:
pipeline_encoded_train = basic_model.named_steps[
    'preprocessor'
].transform(X_train)

print("Pipeline encoded shape:", pipeline_encoded_train.shape)

Pipeline encoded shape: (5917, 234)


In [52]:
X_old = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

X_old_train = X_old.loc[X_train.index]
X_old_test = X_old.loc[X_test.index]

In [53]:
lr_old = LinearRegression()

lr_old.fit(X_old_train, y_train)

pred_old = lr_old.predict(X_old_test)

print("Old method R2:",
      metrics.r2_score(y_test, pred_old))

Old method R2: 0.8430237692980013


In [54]:
import joblib

joblib.dump(basic_model, "basic_model.pkl")

['basic_model.pkl']

In [55]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [56]:
decision_tree_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(
        random_state=42
    ))
])

decision_tree_model.fit(X_train, y_train)

y_pred_dt = decision_tree_model.predict(X_test)

r2_dt = r2_score(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print("Decision Tree Results")
print("---------------------")
print("R²   :", r2_dt)
print("MAE  :", mae_dt)
print("RMSE :", rmse_dt)

Decision Tree Results
---------------------
R²   : 0.8687254775116904
MAE  : 57707.17635135135
RMSE : 101911.74302120239


In [57]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest Results")
print("---------------------")
print("R²   :", r2_rf)
print("MAE  :", mae_rf)
print("RMSE :", rmse_rf)

Random Forest Results
---------------------
R²   : 0.9214333762766488
MAE  : 43871.01752702703
RMSE : 78841.16828154464


In [58]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

gradient_boosting_model.fit(X_train, y_train)

y_pred_gb = gradient_boosting_model.predict(X_test)

r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print("Gradient Boosting Results")
print("-------------------------")
print("R²   :", r2_gb)
print("MAE  :", mae_gb)
print("RMSE :", rmse_gb)

Gradient Boosting Results
-------------------------
R²   : 0.8595263177185422
MAE  : 68288.45222386866
RMSE : 105422.05782287069


In [59]:
comparison = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "R2": [
        r2_dt,
        r2_rf,
        r2_gb
    ],
    "MAE": [
        mae_dt,
        mae_rf,
        mae_gb
    ],
    "RMSE": [
        rmse_dt,
        rmse_rf,
        rmse_gb
    ]
})

comparison.sort_values("R2", ascending=False)

,Model,R2,MAE,RMSE
1,Random Forest,0.921433,43871.017527,78841.168282
0,Decision Tree,0.868725,57707.176351,101911.743021
2,Gradient Boosting,0.859526,68288.452224,105422.057823


In [60]:
from sklearn.model_selection import GridSearchCV

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation R²:")
print(grid_search.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters:
{'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}

Best Cross-Validation R²:
0.8913915194134219


In [61]:
best_rf_model = grid_search.best_estimator_

y_pred_tuned_rf = best_rf_model.predict(X_test)

r2_tuned_rf = r2_score(y_test, y_pred_tuned_rf)
mae_tuned_rf = mean_absolute_error(y_test, y_pred_tuned_rf)
rmse_tuned_rf = np.sqrt(mean_squared_error(y_test, y_pred_tuned_rf))

print("Tuned Random Forest Results")
print("---------------------------")
print("R²   :", r2_tuned_rf)
print("MAE  :", mae_tuned_rf)
print("RMSE :", rmse_tuned_rf)

Tuned Random Forest Results
---------------------------
R²   : 0.9206786259583131
MAE  : 43820.70608108108
RMSE : 79218.95700864302


In [62]:
import joblib

final_basic_model = grid_search.best_estimator_

joblib.dump(final_basic_model, "basic_model_v2.pkl")

print("Basic model saved successfully!")


Basic model saved successfully!
